In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.bias import Balance
from dataeval.core import compute_stats
from dataeval.data import Indices, Limit, Shuffle, View
from dataeval.exceptions import MetadataFormatError
from dataeval.flags import ImageStats

In [3]:
dataset = View(SeaDrone(root="./data", image_set="val", download=True), [Shuffle(seed=0), Limit(200)])
metadata = Metadata(dataset)

print("levels      :", metadata.levels)
print("level counts:", metadata.level_counts)

levels      : ('unit', 'instance')
level counts: {'unit': 200, 'instance': 1305}


/tmp/ipykernel_5861/2048650437.py:4: UserWarning: `date_time`, `latitude` and `longitude` were dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  print("levels      :", metadata.levels)


In [4]:
EXPECTED_COUNTS = {"unit": 200, "instance": 1305}

if dict(metadata.level_counts) != EXPECTED_COUNTS:
    raise AssertionError(
        f"This tutorial's prose was written against {EXPECTED_COUNTS} (maite-datasets 0.0.18), "
        f"but your dataset yields {dict(metadata.level_counts)}. Every idea below still holds — "
        "levels, weighting, aggregation — but the specific figures quoted will not match your output."
    )

In [5]:
print("one row per image:")
print(metadata.rows_at("unit").select("item_index", "altitude", "speed").head(3))

print("\none row per detection:")
print(metadata.rows_at("instance").select("item_index", "class_label", "object_size").head(3))

one row per image:
shape: (3, 3)
┌────────────┬───────────┬───────┐
│ item_index ┆ altitude  ┆ speed │
│ ---        ┆ ---       ┆ ---   │
│ i64        ┆ f64       ┆ f64   │
╞════════════╪═══════════╪═══════╡
│ 0          ┆ 17.099166 ┆ 0.0   │
│ 1          ┆ 9.099556  ┆ 0.0   │
│ 2          ┆ -1.0      ┆ -1.0  │
└────────────┴───────────┴───────┘

one row per detection:
shape: (3, 3)
┌────────────┬─────────────┬─────────────┐
│ item_index ┆ class_label ┆ object_size │
│ ---        ┆ ---         ┆ ---         │
│ i64        ┆ i64         ┆ i64         │
╞════════════╪═════════════╪═════════════╡
│ 0          ┆ 2           ┆ 782         │
│ 0          ┆ 2           ┆ 2916        │
│ 0          ┆ 2           ┆ 1392        │
└────────────┴─────────────┴─────────────┘


In [6]:
valid = pl.col("altitude") >= 0
per_image = metadata.rows_at("unit").filter(valid)["altitude"]
per_detection = metadata.rows_at("instance").filter(valid)["altitude"]

print(f"per image     n={len(per_image):5d}  mean={per_image.mean():.1f} m  std={per_image.std():.1f}")
print(f"per detection n={len(per_detection):5d}  mean={per_detection.mean():.1f} m  std={per_detection.std():.1f}")

per image     n=  142  mean=41.8 m  std=29.8
per detection n=  922  mean=48.1 m  std=30.8


In [7]:
print("factor_data at instance (default):", metadata.factor_data.shape)
print("factor_data at unit               :", metadata.at("unit").factor_data.shape)

factor_data at instance (default): (1305, 17)
factor_data at unit               : (200, 15)


/tmp/ipykernel_5861/1016173203.py:1: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `id`, `image_id`, `object_id`, `object_size`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  print("factor_data at instance (default):", metadata.factor_data.shape)


In [8]:
try:
    labels_per_image = metadata.at("unit").class_labels
except ValueError as error:
    print("as expected:", str(error).split(".")[0])

as expected: class_labels is defined at the 'instance' level, but this metadata is viewed at 'unit', which has no label per row


In [9]:
own_only = Metadata(dataset, inherited=False)

print("at instance, inherited=True :", len(metadata.factor_names), "factors")
print("at instance, inherited=False:", sorted(own_only.factor_names))

at instance, inherited=True : 17 factors


at instance, inherited=False: ['object_id', 'object_size']


In [10]:
stats = compute_stats(
    dataset,
    stats=ImageStats.VISUAL_SHARPNESS | ImageStats.VISUAL_BRIGHTNESS,
    per_image=True,
    per_target=True,
    per_background=True,
    normalize_pixel_values=False,
)

print("measured:", sorted(stats["stats"]))

measured: ['background_brightness', 'background_fraction', 'background_sharpness', 'brightness', 'sharpness']


In [11]:
extrinsic = set(metadata.factor_names)
metadata.add_factors(stats)

print("new factors:", sorted(set(metadata.factor_names) - extrinsic))
print("dropped    :", metadata.dropped_factors)

new factors: ['instance_brightness', 'instance_sharpness', 'unit_background_brightness', 'unit_background_fraction', 'unit_background_sharpness', 'unit_brightness', 'unit_sharpness']
dropped    : {'latitude': ['cardinality_over_budget'], 'date_time': ['cardinality_over_budget'], 'longitude': ['cardinality_over_budget'], 'instance_background_brightness': ['no_values_at_level'], 'instance_background_sharpness': ['no_values_at_level'], 'instance_background_fraction': ['no_values_at_level']}


In [12]:
print(
    metadata
    .rows_at("unit")
    .select("item_index", "unit_sharpness", "unit_background_sharpness", "instance_sharpness")
    .head(3)
)

print("\nand from the detection rows, where the image value propagates down:")
print(metadata.rows_at("instance").select("item_index", "unit_sharpness", "instance_sharpness").head(3))

shape: (3, 4)
┌────────────┬────────────────┬───────────────────────────┬────────────────────┐
│ item_index ┆ unit_sharpness ┆ unit_background_sharpness ┆ instance_sharpness │
│ ---        ┆ ---            ┆ ---                       ┆ ---                │
│ i64        ┆ f32            ┆ f32                       ┆ f32                │
╞════════════╪════════════════╪═══════════════════════════╪════════════════════╡
│ 0          ┆ 10.87645       ┆ 10.781616                 ┆ null               │
│ 1          ┆ 22.498152      ┆ 19.834988                 ┆ null               │
│ 2          ┆ 7.374899       ┆ 7.114103                  ┆ null               │
└────────────┴────────────────┴───────────────────────────┴────────────────────┘

and from the detection rows, where the image value propagates down:
shape: (3, 3)
┌────────────┬────────────────┬────────────────────┐
│ item_index ┆ unit_sharpness ┆ instance_sharpness │
│ ---        ┆ ---            ┆ ---                │
│ i64        ┆ 

In [13]:
units = metadata.rows_at("unit")
print(
    units.select(
        pl.col("unit_background_fraction").min().alias("smallest"),
        pl.col("unit_background_fraction").median().alias("median"),
    )
)

shape: (1, 2)
┌──────────┬──────────┐
│ smallest ┆ median   │
│ ---      ┆ ---      │
│ f32      ┆ f32      │
╞══════════╪══════════╡
│ 0.800187 ┆ 0.994466 │
└──────────┴──────────┘


In [14]:
print(f"sharpness  object     : {metadata.rows_at('instance')['instance_sharpness'].mean():.1f}")
print(f"           whole image: {units['unit_sharpness'].mean():.1f}")
print(f"           background : {units['unit_background_sharpness'].mean():.1f}")

sharpness  object     : 41.1
           whole image: 16.5
           background : 15.8


In [15]:
sharper = units.select((pl.col("unit_sharpness") > pl.col("unit_background_sharpness")).sum()).item()
print(f"images sharper as a whole than their own background: {sharper} / {len(units)}")

images sharper as a whole than their own background: 197 / 200


In [16]:
gap = (pl.col("unit_brightness") - pl.col("unit_background_brightness")).abs()

print(f"brightness whole image: {units['unit_brightness'].mean():.1f}")
print(f"           background : {units['unit_background_brightness'].mean():.1f}")
print(f"largest gap on any single image: {units.select(gap.max()).item():.1f}")

brightness whole image: 101.0
           background : 101.0
largest gap on any single image: 2.0


In [17]:
enriched = metadata.agg(
    "instance",
    "unit",
    pl.len().alias("n_objects"),
    pl.col("object_size").mean().alias("mean_object_size"),
    pl.col("instance_sharpness").mean().alias("mean_object_sharpness"),
)

new_factors = sorted(set(enriched.at("unit").factor_names) - set(metadata.at("unit").factor_names))
print("new factors at the image level:", new_factors)
print(enriched.rows_at("unit").select("item_index", "altitude", "n_objects", "mean_object_size").head(5))

new factors at the image level: ['mean_object_sharpness', 'mean_object_size', 'n_objects']
shape: (5, 4)
┌────────────┬───────────┬───────────┬──────────────────┐
│ item_index ┆ altitude  ┆ n_objects ┆ mean_object_size │
│ ---        ┆ ---       ┆ ---       ┆ ---              │
│ i64        ┆ f64       ┆ u32       ┆ f64              │
╞════════════╪═══════════╪═══════════╪══════════════════╡
│ 0          ┆ 17.099166 ┆ 5         ┆ 1126.0           │
│ 1          ┆ 9.099556  ┆ 1         ┆ 226032.0         │
│ 2          ┆ -1.0      ┆ 5         ┆ 12361.0          │
│ 3          ┆ 29.798547 ┆ 3         ┆ 9713.666667      │
│ 4          ┆ 17.099166 ┆ 5         ┆ 1184.0           │
└────────────┴───────────┴───────────┴──────────────────┘


In [18]:
crowding = enriched.rows_at("unit").filter(valid)
print("altitude vs objects-per-image, correlation:")
print(round(float(crowding.select(pl.corr("altitude", "n_objects")).item()), 3))

altitude vs objects-per-image, correlation:
0.333


In [19]:
print(
    enriched
    .rows_at("unit")
    .select("item_index", "n_objects", "unit_sharpness", "unit_background_sharpness", "mean_object_sharpness")
    .head(5)
)

shape: (5, 5)
┌────────────┬───────────┬────────────────┬───────────────────────────┬───────────────────────┐
│ item_index ┆ n_objects ┆ unit_sharpness ┆ unit_background_sharpness ┆ mean_object_sharpness │
│ ---        ┆ ---       ┆ ---            ┆ ---                       ┆ ---                   │
│ i64        ┆ u32       ┆ f32            ┆ f32                       ┆ f32                   │
╞════════════╪═══════════╪════════════════╪═══════════════════════════╪═══════════════════════╡
│ 0          ┆ 5         ┆ 10.87645       ┆ 10.781616                 ┆ 50.692177             │
│ 1          ┆ 1         ┆ 22.498152      ┆ 19.834988                 ┆ 61.729645             │
│ 2          ┆ 5         ┆ 7.374899       ┆ 7.114103                  ┆ 33.115841             │
│ 3          ┆ 3         ┆ 30.256311      ┆ 30.112347                 ┆ 55.972309             │
│ 4          ┆ 5         ┆ 11.442901      ┆ 11.321176                 ┆ 55.266666             │
└────────────┴───────────┴

In [20]:
per_object = Balance().evaluate(metadata)
ranked = per_object.balance.sort("mi_value", descending=True)
print(ranked.head(5))

print("\nobject_size, for comparison:")
print(ranked.filter(pl.col("factor_name") == "object_size"))

/tmp/ipykernel_5861/2948580323.py:1: UserWarning: `instance_brightness`, `instance_sharpness`, `unit_background_brightness`, `unit_background_fraction`, `unit_background_sharpness`, `unit_brightness` and `unit_sharpness` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"instance_brightness": [...]} to control this.
  per_object = Balance().evaluate(metadata)


shape: (5, 2)
┌──────────────┬──────────┐
│ factor_name  ┆ mi_value │
│ ---          ┆ ---      │
│ cat          ┆ f64      │
╞══════════════╪══════════╡
│ class_label  ┆ 1.0      │
│ storage      ┆ 0.317702 │
│ gimbal_pitch ┆ 0.229883 │
│ object_id    ┆ 0.221884 │
│ altitude     ┆ 0.212443 │
└──────────────┴──────────┘

object_size, for comparison:
shape: (1, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ object_size ┆ 0.033163 │
└─────────────┴──────────┘


In [21]:
per_image_balance = Balance(label="n_objects").evaluate(enriched.at("unit"))
print(per_image_balance.balance.sort("mi_value", descending=True).head(5))

/tmp/ipykernel_5861/2151317321.py:1: UserWarning: `mean_object_sharpness`, `mean_object_size`, `unit_background_brightness`, `unit_background_fraction`, `unit_background_sharpness`, `unit_brightness` and `unit_sharpness` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"mean_object_sharpness": [...]} to control this.
  per_image_balance = Balance(label="n_objects").evaluate(enriched.at("unit"))


shape: (5, 2)
┌──────────────────┬──────────┐
│ factor_name      ┆ mi_value │
│ ---              ┆ ---      │
│ cat              ┆ f64      │
╞══════════════════╪══════════╡
│ n_objects        ┆ 1.0      │
│ storage          ┆ 0.49952  │
│ image_id         ┆ 0.294102 │
│ id               ┆ 0.275378 │
│ mean_object_size ┆ 0.265924 │
└──────────────────┴──────────┘


In [22]:
swimmer_rows = metadata.where(pl.col("class_label") == 1, level="instance")
swimmer_images = metadata.having(pl.col("class_label") == 1, level="instance")

print("unfiltered:", metadata.level_counts)
print("where     :", swimmer_rows.level_counts)
print("having    :", swimmer_images.level_counts)

unfiltered: {'unit': 200, 'instance': 1305}
where     : {'unit': 200, 'instance': 907}
having    : {'unit': 178, 'instance': 1235}


In [23]:
high = metadata.where(pl.col("altitude") > 60, level="unit")
print("altitude > 60 m:", high.level_counts)

altitude > 60 m: {'unit': 22, 'instance': 192}


In [24]:
print("is_filtered:", high.is_filtered)

is_filtered: True


In [25]:
items = high.selected_items()
matching_dataset = View(dataset, Indices(items.tolist()))

print("surviving items :", len(items))
print("matching dataset:", len(matching_dataset))

surviving items : 22
matching dataset: 22


In [26]:
try:
    swimmer_rows.selected_items()
except ValueError as error:
    print("as expected:", str(error).split(".")[0])

as expected: This metadata was filtered below the item level, so no subset of the dataset reproduces it: some 'unit' row kept only part of its rows


In [27]:
enriched.save("./data/seadrone-metadata.dem")
reloaded = Metadata.load("./data/seadrone-metadata.dem", dataset)

print("levels     :", reloaded.levels)
print("counts     :", reloaded.level_counts)
print("agg factors:", sorted(set(reloaded.at("unit").factor_names) - set(metadata.at("unit").factor_names)))

levels     : ('unit', 'instance')
counts     : {'unit': 200, 'instance': 1305}
agg factors: ['mean_object_sharpness', 'mean_object_size', 'n_objects']


In [28]:
try:
    metadata = Metadata.load("./data/seadrone-metadata.dem", dataset)
except MetadataFormatError:
    metadata = Metadata(dataset)
    metadata.save("./data/seadrone-metadata.dem")

print("ready:", metadata.level_counts)

ready: {'unit': 200, 'instance': 1305}
